# Notebook 01B — Parser do HTML

Este notebook realiza a leitura do HTML coletado no Notebook 01A e extrai todos os links para documentos da OBB.

Ao final será gerado o arquivo:

metadata_links.csv

In [1]:
# @title Importação das bibliotecas
from pathlib import Path

import pandas as pd

from bs4 import BeautifulSoup

import re

import os

from google.colab import files

In [2]:
# @title Configuração das pastas
DATA_DIR = Path("../data")

HTML_DIR = DATA_DIR / "raw" / "html"

METADATA_DIR = DATA_DIR / "raw" / "metadata"

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
# @title Abrir o HTML
arquivo_html = '/content/drive/MyDrive/Colab Notebooks/OBB_GRA/Data/raw/html/latest.html'
arquivo_html = Path(arquivo_html)

with open(
    arquivo_html,
    encoding="utf-8"
) as f:

    html = f.read()

print(len(html))

94974


In [4]:
# @title Criar o BeautifulSoup
soup = BeautifulSoup(
    html,
    "html.parser"
)

In [5]:
# @title Quantos links existem?
links = soup.find_all("a")

print(f"Foram encontrados {len(links)} links.")

Foram encontrados 168 links.


In [6]:
# @title Filtrar apenas documentos
DOCUMENTOS = (
    ".pdf",
    ".doc",
    ".docx"
)

In [7]:
# @title Extração
registros = []

for a in links:

    href = a.get("href")

    if not href:
        continue

    href = href.strip()

    if not href.lower().endswith(DOCUMENTOS):
        continue

    texto = a.get_text(
        " ",
        strip=True
    )

    registros.append({

        "texto": texto,

        "url": href

    })

In [8]:
# @title Criando DataFrame
df = pd.DataFrame(
    registros
)

df

,texto,url
0,Gabarito definitivo XXII OBB - 3ª fase,https://olimpiadasdebiologia.butantan.gov.br/a...
1,Gabarito definitivo XXII OBB - 2ª fase,https://olimpiadasdebiologia.butantan.gov.br/a...
2,Prova XXII OBB - 2ª fase (Fonte 18),https://olimpiadasdebiologia.butantan.gov.br/a...
3,Prova XXII OBB - 2ª fase,https://olimpiadasdebiologia.butantan.gov.br/a...
4,Gabarito definitivo XXII OBB - 1ª fase,https://olimpiadasdebiologia.butantan.gov.br/a...
...,...,...
89,Gabarito_V_OBB_2afase.doc,https://olimpiadasdebiologia.butantan.gov.br/a...
90,Gabarito_V_OBB_1afase.doc,https://olimpiadasdebiologia.butantan.gov.br/a...
91,Gabarito_II_OBB.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...
92,Gabarito_I_OBB.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...


In [9]:
# @title Remover view-source
df["url"] = (
    df["url"]
      .str.replace(
          "view-source:",
          "",
          regex=False
      )
)

In [10]:
# @title Remover espaços
df["texto"] = (
    df["texto"]
      .str.strip()
)

df["url"] = (
    df["url"]
      .str.strip()
)

In [11]:
# @title Remover duplicados
df = (
    df
    .drop_duplicates(
        subset="url"
    )
    .reset_index(drop=True)
)

In [12]:
# @title Criar ID
df.insert(
    0,
    "id_documento",
    range(1, len(df)+1)
)

In [13]:
# @title Nome do arquivo
df["arquivo"] = (
    df["url"]
    .apply(
        lambda x: Path(x).name
    )
)

In [14]:
# @title Extensão
df["extensao"] = (
    df["arquivo"]
      .str.split(".")
      .str[-1]
      .str.upper()
)

In [15]:
# @title Verificando valores Nulos
df.isnull().sum()

,0
id_documento,0
texto,0
url,0
arquivo,0
extensao,0


In [16]:
# @title Verificando duplicados
df.duplicated().sum()

np.int64(0)

In [17]:
# @title Conferindo o DataFrame
df.sample(10, random_state=42)

,id_documento,texto,url,arquivo,extensao
40,41,Prova da Fase Seletiva/Gabarito Comentado,https://olimpiadasdebiologia.butantan.gov.br/a...,Prova%20e%20gabarito%20-%20Seletiva%20internac...,PDF
22,23,Prova da Fase 1 - fonte 12,https://olimpiadasdebiologia.butantan.gov.br/a...,OBB_documento_fonte%2012.pdf,PDF
55,56,XIII_OBB_1afase.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...,XIII_OBB_1afase.pdf,PDF
88,89,Gabarito_II_OBB.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...,Gabarito_II_OBB.pdf,PDF
0,1,Gabarito definitivo XXII OBB - 3ª fase,https://olimpiadasdebiologia.butantan.gov.br/a...,gabarito-definitivo-prova-fase-3.pdf,PDF
26,27,Gabarito definitivo da fase 2A,https://olimpiadasdebiologia.butantan.gov.br/a...,XIX_OBB-gabarito-definitivo-2a.pdf,PDF
39,40,Gabarito Definitivo da Fase 1 OBB 2021,https://olimpiadasdebiologia.butantan.gov.br/a...,XVII%20OBB_gabarito_definitivo_fase_1.pdf,PDF
66,67,VII_OBB_2afase.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...,VII_OBB_2afase.pdf,PDF
10,11,Prova da Fase 2 - fonte 18,https://olimpiadasdebiologia.butantan.gov.br/a...,prova-fase2-obb-18pt.pdf,PDF
44,45,Prova XV OBB - 2ª Fase​,https://olimpiadasdebiologia.butantan.gov.br/a...,Porva_fase2_OBB_2019.pdf,PDF


In [18]:
# @title Estatísticas
print(f"Total de documentos : {len(df)}")

print(df["extensao"].value_counts())

Total de documentos : 91
extensao
PDF     84
DOC      5
DOCX     2
Name: count, dtype: int64


In [19]:
# @title Ordenação
df = df.sort_values(
    by="texto"
).reset_index(drop=True)

In [20]:
df

,id_documento,texto,url,arquivo,extensao
0,13,Fase 1 - Lista geral,https://olimpiadasdebiologia.butantan.gov.br/a...,Alunos_OBB_2025-lista-geral.pdf,PDF
1,42,Fase 1 da OBB 2020,https://olimpiadasdebiologia.butantan.gov.br/a...,Prova_fase1_OBB_2020.pdf,PDF
2,39,Fase 1 da OBB 2021,https://olimpiadasdebiologia.butantan.gov.br/a...,XVII%20OBB%20_prova_fase_1_v_final.pdf,PDF
3,9,Fase 2 - Lista geral,https://olimpiadasdebiologia.butantan.gov.br/a...,OBB_Alunos_Fase_2_Geral.pdf,PDF
4,40,Gabarito Definitivo da Fase 1 OBB 2021,https://olimpiadasdebiologia.butantan.gov.br/a...,XVII%20OBB_gabarito_definitivo_fase_1.pdf,PDF
...,...,...,...,...,...
86,60,XI_OBB XI_1afase.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...,XI_OBB%20XI_1afase.pdf,PDF
87,59,XI_OBB_2afase.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...,XI_OBB_2afase.pdf,PDF
88,64,X_OBB_1afase.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...,IX_OBB_1afase.pdf,PDF
89,62,X_OBB_1afase.pdf,https://olimpiadasdebiologia.butantan.gov.br/a...,X_OBB_1afase.pdf,PDF


In [21]:
df.loc[
    df["extensao"] != "PDF",
    ["texto", "arquivo", "url"]
]

,texto,arquivo,url
21,Gabarito_IX_OBB_1afase.docx,Gabarito_IX_OBB_1afase.docx,https://olimpiadasdebiologia.butantan.gov.br/a...
22,Gabarito_IX_OBB_2afase.docx,Gabarito_IX_OBB_2afase.docx,https://olimpiadasdebiologia.butantan.gov.br/a...
24,Gabarito_VIII_OBB_1afase.doc,Gabarito_VIII_OBB_1afase.doc,https://olimpiadasdebiologia.butantan.gov.br/a...
25,Gabarito_VII_OBB_2afase.doc,Gabarito_VII_OBB_2afase.doc,https://olimpiadasdebiologia.butantan.gov.br/a...
26,Gabarito_VI_OBB_2afase.doc,Gabarito_VI_OBB_2afase.doc,https://olimpiadasdebiologia.butantan.gov.br/a...
27,Gabarito_V_OBB_1afase.doc,Gabarito_V_OBB_1afase.doc,https://olimpiadasdebiologia.butantan.gov.br/a...
28,Gabarito_V_OBB_2afase.doc,Gabarito_V_OBB_2afase.doc,https://olimpiadasdebiologia.butantan.gov.br/a...


In [22]:
df["extensao"].value_counts(normalize=True) * 100

,proportion
extensao,
PDF,92.307692
DOC,5.494505
DOCX,2.197802


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_documento  91 non-null     int64 
 1   texto         91 non-null     object
 2   url           91 non-null     object
 3   arquivo       91 non-null     object
 4   extensao      91 non-null     object
dtypes: int64(1), object(4)
memory usage: 3.7+ KB


In [24]:
# @title Validação das URLs
df["url"].str.startswith("https://").value_counts()

,count
url,
True,90
False,1


In [25]:

# @title Encontrar a URL inválida
df[df["url"].str.startswith("https://") == False]


,id_documento,texto,url,arquivo,extensao
50,51,Prova XIV OBB - 2ª Fase,http://olimpiadasdebiologia.butantan.gov.br/as...,Prova_fase2_OBB_2018.pdf,PDF


In [26]:
# @title Verificar o conteúdo da URL
url = df.loc[
    ~df["url"].str.startswith("https://"),
    "url"
].iloc[0]

print(repr(url))

'http://olimpiadasdebiologia.butantan.gov.br/assets/arquivos/provas_gabaritos_classificacoes/Prova_fase2_OBB_2018.pdf'


In [27]:
# @title Padronizar urls
df["url"] = df["url"].str.replace(
    "http://",
    "https://",
    regex=False
)

In [28]:
# @title Validar novamente
df["url"].str.startswith("https://").value_counts()

,count
url,
True,91


In [29]:
# @title Salvar o DataFrame

METADATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/OBB_GRA/Data/raw/metadata")

METADATA_DIR.mkdir(parents=True, exist_ok=True)

In [30]:
arquivo_saida = METADATA_DIR / "metadata_links_raw.csv"

df.to_csv(
    arquivo_saida,
    index=False,
    encoding="utf-8-sig"
)

print(f"Arquivo salvo em:\n{arquivo_saida}")

Arquivo salvo em:
/content/drive/MyDrive/Colab Notebooks/OBB_GRA/Data/raw/metadata/metadata_links_raw.csv


In [31]:
# @title Verificar se o arquivo foi salvo
os.listdir(METADATA_DIR)

['metadata_links_raw.csv']

In [32]:
df_salvo = pd.read_csv(
    arquivo_saida,
    encoding="utf-8-sig"
)

df_salvo.head()

,id_documento,texto,url,arquivo,extensao
0,13,Fase 1 - Lista geral,https://olimpiadasdebiologia.butantan.gov.br/a...,Alunos_OBB_2025-lista-geral.pdf,PDF
1,42,Fase 1 da OBB 2020,https://olimpiadasdebiologia.butantan.gov.br/a...,Prova_fase1_OBB_2020.pdf,PDF
2,39,Fase 1 da OBB 2021,https://olimpiadasdebiologia.butantan.gov.br/a...,XVII%20OBB%20_prova_fase_1_v_final.pdf,PDF
3,9,Fase 2 - Lista geral,https://olimpiadasdebiologia.butantan.gov.br/a...,OBB_Alunos_Fase_2_Geral.pdf,PDF
4,40,Gabarito Definitivo da Fase 1 OBB 2021,https://olimpiadasdebiologia.butantan.gov.br/a...,XVII%20OBB_gabarito_definitivo_fase_1.pdf,PDF


In [33]:
# @title Fazer o download para o computador
files.download( str(arquivo_saida))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>